# Notebook 02 — Feature Audit

**Purpose**: Understand which features matter and verify no leakage exists.

This notebook:
1. Loads feature importance from a trained model
2. Visualises top features
3. Computes feature correlation matrix (detect multicollinearity)
4. Checks that lag features don't contain future information
5. Plots rolling feature values over time for sanity checks

**Run after**: `make train EXP=configs/experiments/exp001_lgbm_baseline.yaml`
**Data sources**: `artifacts/exp001_lgbm_baseline/`, `data/processed/features.parquet`

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import sys; sys.path.insert(0, str(Path('../src')))
from janestreet_forecasting.data import schemas as S
from janestreet_forecasting.paths import ARTIFACTS_DIR, PROCESSED_DIR

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

EXPERIMENT_ID = 'exp001_lgbm_baseline'
ARTIFACT_DIR = ARTIFACTS_DIR / EXPERIMENT_ID

## 1. Feature Importance

In [ ]:
fi_path = ARTIFACT_DIR / 'feature_importance.parquet'
if not fi_path.exists():
    print(f'Feature importance not found. Run `make train` first.')
else:
    fi_df = pl.read_parquet(fi_path)
    top30 = fi_df.head(30)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(range(len(top30)), top30['importance_gain'].to_numpy()[::-1], color='steelblue', alpha=0.8)
    ax.set_yticks(range(len(top30)))
    ax.set_yticklabels(top30['feature'].to_list()[::-1], fontsize=9)
    ax.set_xlabel('Mean Gain')
    ax.set_title(f'{EXPERIMENT_ID} — Top 30 Features by Gain')
    plt.tight_layout()
    plt.show()

    # What fraction of importance do the top features hold?
    total = fi_df['importance_gain'].sum()
    top10_pct = fi_df.head(10)['importance_gain'].sum() / total * 100
    print(f'Top 10 features hold {top10_pct:.1f}% of total importance')

## 2. Feature Correlation Matrix (Top Features)

In [ ]:
features_path = PROCESSED_DIR / 'features.parquet'
if not features_path.exists():
    print('Features not found. Run `make build-features` first.')
else:
    df = pl.read_parquet(features_path)

    # Use top features from importance if available, else first 20 raw features
    if fi_path.exists():
        fi_df = pl.read_parquet(fi_path)
        top_cols = [c for c in fi_df.head(20)['feature'].to_list() if c in df.columns]
    else:
        top_cols = [c for c in S.FEATURE_COLS[:20] if c in df.columns]

    sample = df.sample(min(50_000, len(df)), seed=42)
    corr_data = sample.select(top_cols).to_pandas().corr()

    mask = np.triu(np.ones_like(corr_data), k=1)
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(
        corr_data, mask=mask, annot=False, fmt='.2f',
        cmap='RdBu_r', center=0, vmin=-1, vmax=1,
        square=True, linewidths=0.5, ax=ax
    )
    ax.set_title('Feature Correlation Matrix (Top 20 by Importance)')
    plt.tight_layout()
    plt.show()

## 3. Leakage Check: Lag Feature Values Over Time

In [ ]:
# The lag_0 column for responder_6 should be the PREVIOUS time step's responder_6.
# If lag correlation > 0.99, something is wrong (data leaked).

if features_path.exists():
    df = pl.read_parquet(features_path)

    if S.TARGET_COL in df.columns and 'responder_6_lag_0' in df.columns:
        # Compute Pearson correlation between target and its lag
        # If this is exactly 1.0 — severe leakage!
        corr = df.select([
            pl.corr(S.TARGET_COL, 'responder_6_lag_0').alias('corr')
        ])['corr'][0]
        print(f'Correlation between responder_6 and responder_6_lag_0: {corr:.4f}')
        if corr is not None and abs(corr) > 0.99:
            print('⚠️  WARNING: Correlation near 1.0 — possible lag feature leakage!')
        else:
            print('✓ Lag correlation is in normal range (expected small positive value)')